In [1]:
import pandas

/Users/karthickkumarasamy/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv('Detailed_Polling_Data.csv')

<IPython.core.display.Javascript object>

In [3]:
df.columns

Index(['serial no. of polling station',
       'all india anna dravida munnetra kazhagam', 'dravida munnetra kazhagam',
       'naam tamilar katchi', 'tamilaga vettri kazhagam',
       'tamizhaga vaazhvurimai katchi', 'vishwa tamil kazhagam', 'independent',
       'independent.1', 'independent.2', 'independent.3', 'independent.4',
       'independent.5', 'independent.6', 'independent.7',
       'total of valid votes', 'no. of rejected votes', 'nota', 'total',
       'no. of tendered votes', 'polling station number',
       'building in which it will be located', 'polling areas',
       'whether for all voters or men only or women only', 'building_clean',
       'location_clean', 'Winner_Party', 'Winner_Votes', 'Runner_Up_Votes',
       'Margin_Of_Victory', 'Runner_Up_Party', 'AIADMK_Rank', 'DMK_Rank',
       'NTK_Rank', 'TVK_Rank', 'TAVAK_Rank', 'VTK_Rank', 'NOTA_Rank',
       'IND1_Rank', 'IND2_Rank', 'IND3_Rank', 'IND4_Rank', 'IND5_Rank',
       'IND6_Rank', 'IND7_Rank', 'IND8_Rank',

In [4]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the new dataset (Update the filename to match your file)
df6 = pd.read_csv("Detailed_Polling_Data.csv")

# 2. Select the core political parties present in this dataset index
core_parties = [
    'all india anna dravida munnetra kazhagam', 
    'dravida munnetra kazhagam', 
    'naam tamilar katchi', 
    'tamilaga vettri kazhagam'
]

# Ensure zero issues with missing numbers by filling with 0
df6[core_parties] = df6[core_parties].fillna(0)

# 3. Calculate true total votes for normalization (Core + Independents + NOTA)
df6['Total_Calculated_Votes'] = df6[core_parties].sum(axis=1) + df6['Total_Independent_Votes'].fillna(0) + df6['nota'].fillna(0)

# Filter out empty entries to completely avoid division by zero errors
df6 = df6[df6['Total_Calculated_Votes'] > 0].copy()

# 4. Feature Engineering: Create normalized percentage shares (%)
share_cols = []
for party in core_parties:
    col_name = f'{party}_share_pct'
    df6[col_name] = (df6[party] / df6['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)

# Append strategic structural dimensions
df6['independent_share_pct'] = (df6['Total_Independent_Votes'].fillna(0) / df6['Total_Calculated_Votes']) * 100
feature_cols = share_cols + ['independent_share_pct', 'Margin_Percentage']

# Drop or fill edge-case missing numbers inside target features
df6[feature_cols] = df6[feature_cols].fillna(0)

# 5. Extract and Scale features for the ML model
X = df6[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df6['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 7. Print the Raw Profile Breakdown to help map the text identities
print("\n--- DATASET 6: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
profile = df6.groupby('Cluster_ID')[feature_cols].mean()
print(profile.round(2))

print("\n--- DATASET 6: BOOTH COUNT PER CLUSTER ---")
print(df6['Cluster_ID'].value_counts())

# 8. Export individual target files for campaign ground teams
for cluster_num in range(optimal_k):
    cluster_df = df6[df6['Cluster_ID'] == cluster_num][
        [
            'serial no. of polling station', 
            'location_clean', 
            'building_clean', 
            'polling areas', 
            'Winner_Party', 
            'Margin_Percentage'
        ]
    ]
    filename = f"Dataset_6_Cluster_{cluster_num}_Booths.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated for all 4 clusters.")



--- DATASET 6: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            all india anna dravida munnetra kazhagam_share_pct  \
Cluster_ID                                                       
0                                                       16.79    
1                                                       15.68    
2                                                       24.01    
3                                                       18.19    

            dravida munnetra kazhagam_share_pct  \
Cluster_ID                                        
0                                         30.40   
1                                         37.89   
2                                         34.05   
3                                         47.44   

            naam tamilar katchi_share_pct  tamilaga vettri kazhagam_share_pct  \
Cluster_ID                                                                      
0                                    4.54                               4